# 06 — Steane [[7, 1, 3]] code

Phase 5 demo. The CSS code from the [7, 4] Hamming parity-check matrix. We show how the syndrome literally *is* the Hamming syndrome (read as a binary integer = the bad qubit's 1-indexed position), confirm distance 3 by exhaustive enumeration, and plot the logical-error scaling.

Read alongside `notes/08-steane-7qubit.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qec.pauli import Pauli
from qec.codes import steane

plt.rcParams["figure.figsize"] = (6, 4)


## 1. The Hamming parity-check matrix and the stabilizers

The X-stabilizers are the rows of `H` (with 1 -> X, 0 -> I); the Z-stabilizers are the same rows with 1 -> Z.


In [ ]:
print("HAMMING_H =")
print(steane.HAMMING_H)
print()
print("X-stabilizers:", steane.X_STABILIZERS_STR)
print("Z-stabilizers:", steane.Z_STABILIZERS_STR)
print("logical X    :", steane.LOGICAL_X)
print("logical Z    :", steane.LOGICAL_Z)


## 2. Syndrome of a single Z error literally points at the qubit

For a Z error on qubit `q`, the X-stabilizer syndrome read as a 3-bit integer (s2 s1 s0) equals `q + 1`.


In [ ]:
print(f"{'Z on qubit':>12} | {'sx (binary)':>14} | {'sx (int)':>10} | should be q+1")
print("-" * 60)
for q in range(7):
    x = np.zeros(7, dtype=np.int8); z = np.zeros(7, dtype=np.int8); z[q] = 1
    E = Pauli(x, z, 0)
    sx, sz = steane.syndrome_for_error(E)
    print(f"{q:>12} | {sx:>14b} | {sx:>10d} | {q+1}")
    assert sx == q + 1
print("OK")


## 3. Every weight-1 Pauli is corrected

7 qubits × 3 Paulis = 21 weight-1 errors. All correctable.


In [ ]:
ok = 0
for E in steane._enumerate_paulis(1):
    residual = steane.correct(E)
    lx, lz = steane.is_logical_failure(residual)
    if not lx and not lz and residual.weight() == 0:
        ok += 1
print(f"weight-1 errors corrected: {ok}/21")
assert ok == 21
print("OK")


## 4. Some weight-2 errors are uncorrectable (distance 3, not 5)

In [ ]:
failures = 0
total = 0
for E in steane._enumerate_paulis(2):
    total += 1
    residual = steane.correct(E)
    lx, lz = steane.is_logical_failure(residual)
    if lx or lz:
        failures += 1
print(f"weight-2 patterns that cause logical failure: {failures}/{total}")
assert failures > 0


## 5. Logical error rate, log-log slope ~ 2

In [ ]:
rng = np.random.default_rng(2026)
ps = np.array([0.005, 0.01, 0.02, 0.03, 0.05, 0.07])
shots = 30_000
pls = np.array([steane.monte_carlo_logical_error(p, shots, rng=rng) for p in ps])
print("p     P_L")
for p, pl in zip(ps, pls):
    print(f"{p:.3f}  {pl:.4f}")

mask = pls > 0
slope = np.polyfit(np.log(ps[mask]), np.log(pls[mask]), 1)[0]
print(f"log-log slope: {slope:.2f}  (theory ~ 2)")

plt.loglog(ps, pls, "C2o-", label="Steane [[7,1,3]] (MC)")
plt.loglog(ps, ps, "k:", label="no encoding")
ref = pls[mask][0] * (ps / ps[mask][0])**2
plt.loglog(ps, ref, "C2--", alpha=0.5, label="slope-2 reference")
plt.xlabel("p (per-qubit Pauli error rate)")
plt.ylabel("P_L(p)")
plt.title("Steane code: logical error rate vs physical")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()


## What this notebook gates

- Stabilizers are the rows of `HAMMING_H`, valid CSS code.
- Single-Z syndrome read as binary == bad qubit (1-indexed). Same for single-X with sz.
- Weight-1 errors all corrected; some weight-2 errors fail (distance 3).
- `P_L ~ p^2` below threshold.

Phase 5 complete. Next: notes 09 (CSS theory) and 10 (fault tolerance).
